testing code for half-filling first. let's see if i get same results as my other code

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import slave_rotor_generic
from slave_rotor_atomic import build_rotor_precomp, eval_rotor_obs, solve_sc
from slave_rotor_MF import _DOS_precomp, _spinon_integrals
from slave_rotor_isotropic import _T0_density_table, _lookup_T0_density_table
import importlib
import mpl_style
import os
importlib.reload(mpl_style)
panel_figsize_for, PANEL_FIGSIZE, color_of = mpl_style.apply_style(single_col_width_in=7.06, text_height_in=9.30)

In [23]:
DENS    = 1./6
N       = 6.0
# ------------------------------------------------------------------
# Build T=0 density table (shared between all alpha values)
# ------------------------------------------------------------------
t0_table = _T0_density_table(
        t_perp=0.,
        density_grid=np.linspace(0.0, 1.0, 2001),
        N_e=4000,
        )

# ------------------------------------------------------------------
# Solver settings (shared)
# ------------------------------------------------------------------
SHARED_PARS = {
        'density':    DENS,
        'N':          N,
        'beta':       1000.,    # finite T for fast demonstration
        't_perp':     0.25,
        'K_init':     2.0,
        'M_trunc':    4,        # keep 3D Hilbert space small: dim=(13)^3=2197
        'mixing':     0.25,
        'iterations': 400,
        'tol':        1e-8,
        'h_window':   20.0,
        'eps_window': 20.0,
        'n_coarse':   51,
        'n_eigs':     30,
        'verbose':    0,
        't0_table':   t0_table,
    }

# ------------------------------------------------------------------
# Run scans for three alpha values
# ------------------------------------------------------------------
alphas      = [0.9]
Us_scan = np.linspace(4,10,num=5,endpoint = True)
Zs_by_alpha = {}
Qs_by_alpha = {}

for alpha in alphas:
    print()
    print("=" * 65)
    print(f"  alpha = {alpha}  (beta=100, n={DENS:.4f}, flat DOS)")
    print(f"  Hilbert-space dim = {(2*SHARED_PARS['M_trunc']+1)**(1 if alpha==1.0 else 3)}")
    #print(f"  Linearised U_c = {Uc_theory:.4f}")
    print("=" * 65)
    print(f"  {'U':>6}  {'Z':>9}  {'Q':>10}  {'K':>10}  {'eps_0':>10}  {'h':>10}  conv")
    Zs = []
    Qs = []
    for U in Us_scan:
        pars = dict(SHARED_PARS)
        pars['U']     = U
        pars['alpha'] = alpha
        r = slave_rotor_generic.solve_generic_MF(pars)
        Zs.append(r['Z'])
        Qs.append(r['Q'])
        c = "ok" if r['converged'] else "NO"
        Qv = r['Q_per_valley']
        Qv_str = (
                f"  Q_eta=[{Qv[0]:+.4f},{Qv[1]:+.4f},{Qv[2]:+.4f}]"
                if Qv is not None else ""
            )
        print(
                f"  {U:6.2f}  {r['Z']:9.5f}  {r['Q']:+10.5f}  "
                f"{r['K']:+10.5f}  {r['eps_0']:+10.5f}  {r['h']:+10.5f}  "
                f"{c}{Qv_str}"
            )

        Zs_by_alpha[alpha] = np.array(Zs)
        Qs_by_alpha[alpha] = np.array(Qs)

    # ------------------------------------------------------------------
    # Summary: estimated U_c for each alpha
    # ------------------------------------------------------------------
    #print()
    #print("=" * 65)
    #print("SUMMARY  (last metallic U, threshold Z > 0.01)")
    #p##rint("=" * 65)
    #print(f"  Linearised U_c (theory, all alpha) = {Uc_theory:.4f}")
    #for alpha in alphas:
    #    Zs    = Zs_by_alpha[alpha]
    #    metal = [U for U, Z in zip(Us_scan, Zs) if Z > 0.01]
    #    Uc_est = max(metal) if metal else float('nan')
    #    print(f"  alpha={alpha}  estimated U_c ~ {Uc_est:.3f}")

    # ------------------------------------------------------------------
    # Plot: Z vs U for each alpha
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    colors = {1.0: 'C0', 0.5: 'C1', 0.0: 'C2'}
    marks  = {1.0: 'o', 0.5: 's', 0.0: '^'}

    for ax_idx, ax in enumerate(axes):
        for alpha in alphas:
            Ys = Zs_by_alpha[alpha] if ax_idx == 0 else Qs_by_alpha[alpha]
            ax.plot(Us_scan, Ys,
                    marker=marks[alpha], ls='--', ms=5,
                    color=color_of[alpha],
                    label=fr'$\alpha={alpha}$')
        #ax.axvline(Uc_theory, color='r', ls=':', lw=1.5,
        #           label=f'$U_c={Uc_theory:.2f}$ (theory)')
        ax.set_xlabel('$U$', fontsize=12)
        ax.set_ylabel('$Z = Q^2$' if ax_idx == 0 else r'$Q = \langle\cos\phi_\eta\rangle$',
                      fontsize=12)
        ax.set_title(
            f'MIT scan  $n={DENS:.3f}$  flat DOS  $\\beta=100$  '
            f'$M_{{\\rm trunc}}={SHARED_PARS["M_trunc"]}$',
            fontsize=10,
        )
        ax.legend(fontsize=10)
        ax.set_ylim(-0.02 if ax_idx == 0 else -0.75,
                    1.05  if ax_idx == 0 else 0.05)

    plt.suptitle(
        f'Slave-rotor generic-alpha MF   '
        f'$N={int(N)}$, $n={DENS:.3f}$, flat DOS',
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    fname = 'Figures/tests/generic_alpha_MIT_scan.pdf'
    #plt.savefig(fname, dpi=150, bbox_inches='tight')
    print(f"\nFigure saved: {fname}")


  alpha = 0.9  (beta=100, n=0.1667, flat DOS)
  Hilbert-space dim = 729
       U          Z           Q           K       eps_0           h  conv
    4.00    0.19995    -0.44716    +0.58059    +7.77069    +7.45736  ok  Q_eta=[-0.4472,-0.4472,-0.4472]
    5.50    0.00000    +0.00000    +0.00000   +10.40161   +10.40000  ok


KeyboardInterrupt: 